# 📚 Fiche d'Examen : Machine Learning

Fiche complète pour l'examen de Machine Learning basée sur les TP.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.io import loadmat
from scipy import optimize


## 1. Chargement des Données


In [ ]:
def load_txt(path, delimiter=','):
    data = np.loadtxt(path, delimiter=delimiter)
    X = data[:, :-1]
    y = data[:, -1]
    return X, y

def load_mat(path):
    data = loadmat(path)
    X = data['X']
    y = data['y'].flatten()
    if y.ndim == 1:
        y = y.reshape(-1, 1)
    return X, y

def load_csv(path):
    df = pd.read_csv(path)
    X = df.iloc[:, :-1].values
    y = df.iloc[:, -1].values
    return X, y


## 2. Prétraitement

**Formule de normalisation:** `X_norm = (X - μ) / σ`


In [ ]:
def feature_normalize(X):
    mu = np.mean(X, axis=0)
    sigma = np.std(X, axis=0)
    return (X - mu) / sigma, mu, sigma

def add_bias(X):
    return np.c_[np.ones((X.shape[0], 1)), X]

def train_val_test_split(X, y, train=0.7, val=0.15):
    m = len(y)
    idx = np.random.permutation(m)
    t_size = int(m * train)
    v_size = int(m * val)
    return (X[idx[:t_size]], y[idx[:t_size]],
            X[idx[t_size:t_size+v_size]], y[idx[t_size:t_size+v_size]],
            X[idx[t_size+v_size:]], y[idx[t_size+v_size:]])


## 3. Régression Linéaire

**Formules :**
* Hypothèse : `h_θ(x) = θ^T * x`
* Coût : `J(θ) = (1/(2m)) * Σ(h_θ(x^(i)) - y^(i))^2`
* Gradient : `∇J(θ) = (1/m) * X^T @ (Xθ - y)`
* Mise à jour : `θ := θ - (α/m) * X^T @ (Xθ - y)`


In [ ]:
def h_linear(theta, X):
    return X @ theta

def compute_cost_linear(X, y, theta):
    m = len(y)
    return (1 / (2 * m)) * np.sum((X @ theta - y) ** 2)

def gradient_descent(X, y, theta, alpha, num_iters):
    m = len(y)
    J_history = []
    for _ in range(num_iters):
        errors = X @ theta - y
        theta -= (alpha / m) * (X.T @ errors)
        J_history.append(compute_cost_linear(X, y, theta))
    return theta, J_history


## 4. Régression Logistique

**Formules :**
* Sigmoïde : `g(z) = 1 / (1 + e^(-z))`
* Hypothèse : `h_θ(x) = g(θ^T * x)`
* Coût : `J(θ) = (-1/m) * Σ[y*log(h) + (1-y)*log(1-h)]`
* Gradient : `∇J(θ) = (1/m) * X^T @ (h_θ(X) - y)`


In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def compute_cost_logistic(X, y, theta):
    m = len(y)
    h = np.clip(sigmoid(X @ theta), 1e-15, 1-1e-15)
    return (-1 / m) * np.sum(y * np.log(h) + (1 - y) * np.log(1 - h))

def gradient_descent_logistic(X, y, theta, alpha, num_iters):
    m = len(y)
    J_history = []
    for _ in range(num_iters):
        h = sigmoid(X @ theta)
        theta -= (alpha / m) * (X.T @ (h - y))
        J_history.append(compute_cost_logistic(X, y, theta))
    return theta, J_history

def predict(theta, X, threshold=0.5):
    return (sigmoid(X @ theta) >= threshold).astype(int)


## 5. Réseaux de Neurones

**Architecture type :** Couche d'entrée → Couche cachée → Couche de sortie
* Exemple TP3 : `400 (20x20 pixels) → 25 (cachée) → 10 (chiffres 0-9)`
* Dimensions : `Theta1 = (hidden_size, input_size + 1)` et `Theta2 = (output_size, hidden_size + 1)`

**Formules :**
* Forward : `z^(2) = Θ^(1) * a^(1)`, `a^(2) = g(z^(2))`, `z^(3) = Θ^(2) * a^(2)`, `a^(3) = g(z^(3))`
* Coût : `J(Θ) = (-1/m) * ΣΣ[y_k*log(h_k)] + (λ/(2m)) * ΣΣ(Θ_ji^(l))^2` (Ne pas régulariser le biais)
* Dérivée Sigmoïde : `g'(z) = g(z) * (1 - g(z))`
* Backward : `δ^(L) = a^(L) - y`, `δ^(l) = (Θ^(l))^T * δ^(l+1) ⊙ g'(z^(l))`, `D^(l) = (1/m) * δ^(l+1) * (a^(l))^T`


In [ ]:
def initialize_weights(input_size, hidden_size, output_size):
    eps = 0.12
    Theta1 = np.random.randn(hidden_size, input_size + 1) * eps
    Theta2 = np.random.randn(output_size, hidden_size + 1) * eps
    return Theta1, Theta2

def sigmoid_gradient(z):
    g = sigmoid(z)
    return g * (1 - g)

def forward_propagation(X, Theta1, Theta2):
    m = X.shape[0]
    a1 = np.c_[np.ones((m, 1)), X]
    z2 = a1 @ Theta1.T
    a2 = sigmoid(z2)
    a2 = np.c_[np.ones((m, 1)), a2]
    z3 = a2 @ Theta2.T
    a3 = sigmoid(z3)
    return a3, z2, a2, z3

def compute_cost_nn(X, y, Theta1, Theta2, lambda_reg=0.1):
    m = X.shape[0]
    a3, _, _, _ = forward_propagation(X, Theta1, Theta2)
    a3 = np.clip(a3, 1e-15, 1-1e-15)
    J = (-1 / m) * np.sum(y * np.log(a3) + (1 - y) * np.log(1 - a3))
    reg = (lambda_reg / (2 * m)) * (np.sum(Theta1[:, 1:]**2) + np.sum(Theta2[:, 1:]**2))
    return J + reg

def backpropagation(X, y, Theta1, Theta2, lambda_reg=0.1):
    m = X.shape[0]
    a1 = np.c_[np.ones((m, 1)), X]
    z2 = a1 @ Theta1.T
    a2 = sigmoid(z2)
    a2 = np.c_[np.ones((m, 1)), a2]
    z3 = a2 @ Theta2.T
    a3 = sigmoid(z3)
    delta3 = a3 - y
    D2 = (1 / m) * (delta3.T @ a2)
    D2[:, 1:] += (lambda_reg / m) * Theta2[:, 1:]
    delta2 = (delta3 @ Theta2[:, 1:]) * sigmoid_gradient(z2)
    D1 = (1 / m) * (delta2.T @ a1)
    D1[:, 1:] += (lambda_reg / m) * Theta1[:, 1:]
    return D1, D2

def train_nn(X, y, input_size, hidden_size, output_size, alpha=0.1, num_iters=100, lambda_reg=0.1):
    Theta1, Theta2 = initialize_weights(input_size, hidden_size, output_size)
    J_history = []
    for _ in range(num_iters):
        J_history.append(compute_cost_nn(X, y, Theta1, Theta2, lambda_reg))
        D1, D2 = backpropagation(X, y, Theta1, Theta2, lambda_reg)
        Theta1 -= alpha * D1
        Theta2 -= alpha * D2
    return Theta1, Theta2, J_history

def predict_nn(X, Theta1, Theta2):
    a3, _, _, _ = forward_propagation(X, Theta1, Theta2)
    return np.argmax(a3, axis=1)


## 6. Optimisation et Évaluation

**Learning Rate (α) :** Tester [0.001, 0.01, 0.1, 1.0].
* Oscille/Explose -> Trop grand
* Décroît lentement -> Trop petit

**Régularisation (λ) :** Évite l'overfitting. Tester [0, 0.01, 0.1, 1, 10].

**Learning Curves :**
* *High Bias (Underfitting) :* J_train et J_val élevés. Solutions : Ajouter des features, Réduire λ.
* *High Variance (Overfitting) :* J_train bas, J_val élevé. Solutions : Augmenter λ, Ajouter des données.


In [ ]:
def compute_accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred) * 100

def confusion_matrix(y_true, y_pred):
    tp = np.sum((y_true == 1) & (y_pred == 1))
    tn = np.sum((y_true == 0) & (y_pred == 0))
    fp = np.sum((y_true == 0) & (y_pred == 1))
    fn = np.sum((y_true == 1) & (y_pred == 0))
    return np.array([[tn, fp], [fn, tp]])


## 7. Exemples d'utilisation (TP1, TP2, TP3)


In [ ]:
# TP1 : Régression Linéaire
# X, y = load_txt('ex1data1.txt')
# X_b = add_bias(X)
# theta = np.zeros(2)
# theta_opt, J_history = gradient_descent(X_b, y, theta, alpha=0.01, num_iters=1500)

# TP2 : Régression Logistique
# X, y = load_txt('ex2data1.txt')
# X_norm, _, _ = feature_normalize(X)
# X_b = add_bias(X_norm)
# theta = np.zeros(3)
# theta_opt, _ = gradient_descent_logistic(X_b, y, theta, alpha=0.1, num_iters=400)

# TP3 : Réseaux de neurones
# X, y = load_mat('ex3data1.mat')
# y_onehot = np.eye(10)[y.astype(int).flatten()]
# Theta1, Theta2, J_history = train_nn(X, y_onehot, 400, 25, 10, alpha=0.1)


## 8. Questions fréquentes et Conseils

**Questions :**
* *Quelle est la dimension de X et y ?* `X.shape = (m, n), y.shape = (m,)`. Après bias : `X_b.shape = (m, n+1)`, `theta.shape = (n+1,)`
* *Pourquoi normaliser ?* Pour que la descente de gradient converge plus vite.
* *Pourquoi ne pas initialiser les poids à 0 dans un NN ?* Tous les neurones d'une couche auraient la même valeur -> symétrie -> pas d'apprentissage.

**Conseils :**
1. Vérifie toujours les dimensions avec `.shape`
2. Trace les courbes de coût pour vérifier la convergence
3. Lis bien les questions avant de répondre (dimensions, type de variables...)
